In [1]:
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import precision_recall_fscore_support

In [2]:
dataset = load_dataset("coastalcph/tydi_xor_rc")

df_train = dataset["train"].to_pandas()
df_validation = dataset["validation"].to_pandas()

#filter data 
langlst = ['ko','ar','te']

df_train_filtered = df_train[df_train['lang'].isin(langlst)]
df_validation_filtered = df_validation[df_validation['lang'].isin(langlst)]

#filter data 
#Training data for each language
df_train_ar = df_train[(df_train['lang'] == "ar")]
df_train_ko = df_train[(df_train['lang'] == "ko")]
df_train_te = df_train[(df_train['lang'] == "te")]

#Validation data for each language
df_validation_ar = df_validation[(df_validation['lang'] == "ar")]
df_validation_ko = df_validation[(df_validation['lang'] == "ko")]
df_validation_te = df_validation[(df_validation['lang'] == "te")]

# df_train_filtered


In [3]:
df_validation_filtered.reset_index()

,index,question,context,lang,answerable,answer_start,answer,answer_inlang
0,0,ఒరెగాన్ రాష్ట్రంలోని అతిపెద్ద నగరం ఏది ?,Portland is the largest city in the U.S. state...,te,True,0,Portland,NaN
1,1,కలరా వ్యాధిని మొదటగా ఏ దేశంలో కనుగొన్నారు ?,"The word cholera is from ""kholera"" from χολή ""...",te,True,99,Indian subcontinent,NaN
2,2,కలరా వ్యాధిని మొదటగా ఏ దేశంలో కనుగొన్నారు ?,Since it became widespread in the 19th century...,te,True,451,England,NaN
3,3,మొదటి ప్రపంచ యుద్ధం ఎప్పుడు మొదలయింది ?,World War I occurred from 1914 to 1918. In ter...,te,True,26,1914,NaN
4,4,మొదటి ప్రపంచ యుద్ధం ఎప్పుడు మొదలయింది ?,"World War I (often abbreviated as WWI or WW1),...",te,True,155,28 July 1914,NaN
...,...,...,...,...,...,...,...,...
1150,3006,2011 జనగణన ప్రకారం రెయ్యలగడ్ద గ్రామములో పురుషు...,Reyyalagadda is a village belonging to Gangara...,te,True,378,37,37
1151,3007,2011 జనాభా లెక్కల ప్రకారం బూతుమిల్లిపాడు గ్రామ...,Boothumillipadu is a village in Gannavaram man...,te,True,308,433,433
1152,3008,2011 జనాభా లెక్కల ప్రకారం మల్లవేముల గ్రామ జనాభ...,Mallavemula is a village belonging to Chagalam...,te,False,-1,1131,1131
1153,3009,2011 నాటికి రష్యా దేశ ప్రధాన మంత్రి ఎవరు?,"Andria Urushadze (; born April 25, 1968) is a ...",te,False,-1,Vladimir Putin,వ్లాదిమిర్ పుతిన్


## Fine-tune one multilingual Transformer QA model

In [4]:
from datasets import Dataset
from transformers import AutoModelForQuestionAnswering, Trainer, TrainingArguments

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased", use_fast=True
)
torch.manual_seed(42)

train_examples = Dataset.from_pandas(df_train_filtered, preserve_index=False)

def prepare_examples(examples):
    tokens = tokenizer(
        examples["question"], examples["context"],
        truncation="only_second", max_length=384, stride=128,
        return_overflowing_tokens=True, return_offsets_mapping=True,
        padding="max_length",
    )
    example_numbers = tokens.pop("overflow_to_sample_mapping")
    offsets = tokens.pop("offset_mapping")
    start_positions = []
    end_positions = []

    for i, example_number in enumerate(example_numbers):
        cls_position = tokens["input_ids"][i].index(tokenizer.cls_token_id)
        start_position = cls_position
        end_position = cls_position

        if examples["answerable"][example_number]:
            answer_start = int(examples["answer_start"][example_number])
            answer_end = answer_start + len(examples["answer"][example_number])
            context_tokens = [
                j for j, part in enumerate(tokens.sequence_ids(i)) if part == 1
            ]
            if (context_tokens
                and offsets[i][context_tokens[0]][0] <= answer_start
                and offsets[i][context_tokens[-1]][1] >= answer_end):
                answer_tokens = [
                    j for j in context_tokens
                    if offsets[i][j][0] < answer_end
                    and offsets[i][j][1] > answer_start
                ]
                if answer_tokens:
                    start_position = answer_tokens[0]
                    end_position = answer_tokens[-1]

        start_positions.append(start_position)
        end_positions.append(end_position)

    tokens["start_positions"] = start_positions
    tokens["end_positions"] = end_positions
    return tokens

train_features = train_examples.map(
    prepare_examples, batched=True,
    remove_columns=train_examples.column_names,
)

model = AutoModelForQuestionAnswering.from_pretrained(
    "bert-base-multilingual-cased"
)
training_args = TrainingArguments(
    output_dir="multilingual-qa", 
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=3e-5,
    seed=42, save_strategy="no", report_to="none",
)
trainer = Trainer(model=model, args=training_args, train_dataset=train_features)
trainer.train()

Map:   0%|          | 0/6335 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

Step,Training Loss
500,2.101912


TrainOutput(global_step=811, training_loss=1.889413892414361, metrics={'train_runtime': 593.6118, 'train_samples_per_second': 10.923, 'train_steps_per_second': 1.366, 'total_flos': 1270686128007168.0, 'train_loss': 1.889413892414361, 'epoch': 1.0})

In [ ]:
# Build a clean validation Dataset from the dataframe you already created.
validation_examples = Dataset.from_pandas(
    df_validation_filtered.reset_index(drop=True),
    preserve_index=False
)

# Add a stable ID if the dataset does not already provide one.
if "id" not in validation_examples.column_names:
    validation_examples = validation_examples.add_column(
        "id", [str(i) for i in range(len(validation_examples))]
    )
else:
    # IDs are converted to strings to make feature/example matching robust.
    validation_examples = validation_examples.map(
        lambda x: {"id": str(x["id"])}
    )

def prepare_validation_features(examples):
    tokens = tokenizer(
        examples["question"],
        examples["context"],
        truncation="only_second",
        max_length=384,
        stride=128,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    example_numbers = tokens.pop("overflow_to_sample_mapping")
    example_ids = []

    for i, example_number in enumerate(example_numbers):
        example_ids.append(examples["id"][example_number])

        # Keep character offsets only for context tokens.
        sequence_ids = tokens.sequence_ids(i)
        tokens["offset_mapping"][i] = [
            offset if sequence_ids[j] == 1 else None
            for j, offset in enumerate(tokens["offset_mapping"][i])
        ]

    tokens["example_id"] = example_ids
    return tokens

validation_features = validation_examples.map(
    prepare_validation_features,
    batched=True,
    remove_columns=validation_examples.column_names,
)


print("Validation examples:", len(validation_examples))
print("Validation features/windows:", len(validation_features))

Map:   0%|          | 0/1155 [00:00<?, ? examples/s]

NameError: name 'example_ids' is not defined

In [6]:
# String IDs and offset mappings are needed for post-processing, but they
# must not be passed into the Transformer itself.
model_validation_features = validation_features.remove_columns(
    ["example_id", "offset_mapping"]
)

validation_output = trainer.predict(model_validation_features)
start_logits, end_logits = validation_output.predictions

print("start logits:", start_logits.shape)
print("end logits:", end_logits.shape)
assert len(start_logits) == len(validation_features)

start logits: (1180, 384)
end logits: (1180, 384)


In [ ]:
import numpy as np
from collections import defaultdict

N_BEST_SIZE = 20
MAX_ANSWER_LENGTH = 64

def make_raw_predictions(
    examples,
    features,
    start_logits,
    end_logits,
    n_best_size=N_BEST_SIZE,
    max_answer_length=MAX_ANSWER_LENGTH,
):
    # Map original example IDs to row indices.
    example_id_to_index = {
        str(example_id): i for i, example_id in enumerate(examples["id"])
    }

    features_per_example = defaultdict(list)
    for feature_index, feature in enumerate(features):
        features_per_example[
            example_id_to_index[str(feature["example_id"])]
        ].append(feature_index)

    rows = []

    for example_index, example in enumerate(examples):
        context = example["context"]

        min_null_score = float("inf")
        best_span_score = -float("inf")
        best_answer = ""
        best_start_char = -1
        best_end_char = -1

        for feature_index in features_per_example[example_index]:
            s_logits = start_logits[feature_index]
            e_logits = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]
            input_ids = features[feature_index]["input_ids"]

            cls_index = input_ids.index(tokenizer.cls_token_id)
            null_score = float(
                s_logits[cls_index] + e_logits[cls_index]
            )
            min_null_score = min(min_null_score, null_score)

            start_indexes = np.argsort(s_logits)[-n_best_size:][::-1]
            end_indexes = np.argsort(e_logits)[-n_best_size:][::-1]

            for start_index in start_indexes:
                for end_index in end_indexes:
                    if start_index >= len(offsets) or end_index >= len(offsets):
                        continue
                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue
                    if end_index < start_index:
                        continue
                    if end_index - start_index + 1 > max_answer_length:
                        continue

                    start_char = offsets[start_index][0]
                    end_char = offsets[end_index][1]

                    if end_char <= start_char:
                        continue

                    span_score = float(
                        s_logits[start_index] + e_logits[end_index]
                    )

                    if span_score > best_span_score:
                        best_span_score = span_score
                        best_answer = context[start_char:end_char]
                        best_start_char = int(start_char)
                        best_end_char = int(end_char)

        if best_span_score == -float("inf"):
            best_span_score = -1e30

        gold_answer = example["answer"] if example["answerable"] else ""

        rows.append({
            "id": str(example["id"]),
            "lang": example["lang"],
            "question": example["question"],
            "context": context,
            "gold_answer": gold_answer,
            "gold_answerable": bool(example["answerable"]),
            "best_non_empty_answer": best_answer,
            "best_non_empty_start": best_start_char,
            "best_non_empty_end": best_end_char,
            "null_score": float(min_null_score),
            "best_non_empty_score": float(best_span_score),
            "no_answer_score": float(min_null_score - best_span_score),
        })

    return pd.DataFrame(rows)

raw_predictions = make_raw_predictions(
    validation_examples,
    validation_features,
    start_logits,
    end_logits,
)

display(raw_predictions.head())
display(raw_predictions["no_answer_score"].describe())

original_index                                                         930
id                                                                     500
lang                                                                    ko
question                                              중국에서 가장 작은 도시는 어디인가?
context                  Yima () is a county-level city of Henan, China...
gold_answer                                                           Yima
gold_answerable                                                       True
best_non_empty_answer                                                 Yima
best_non_empty_start                                                     0
best_non_empty_end                                                       4
null_score                                                        1.951936
best_non_empty_score                                              12.73598
no_answer_score                                                 -10.784044
Name: 500, dtype: object

count    1155.000000
mean       -7.583632
std         5.595784
min       -15.432213
25%       -11.414698
50%        -9.368763
75%        -5.454360
max        13.148607
Name: no_answer_score, dtype: float64

In [ ]:
import re
import string
from collections import Counter
from sklearn.metrics import (
    f1_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

def normalize_answer(text):
    text = str(text).lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def squad_exact_match(prediction, gold):
    return float(
        normalize_answer(prediction) == normalize_answer(gold)
    )

def squad_token_f1(prediction, gold):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(gold).split()

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)

    common = Counter(pred_tokens) & Counter(gold_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)

In [ ]:
def predict_answerability(scores, threshold):
    # 1 = answerable, 0 = unanswerable
    return (np.asarray(scores) <= threshold).astype(int)

def candidate_thresholds(scores):
    values = np.sort(np.unique(np.asarray(scores, dtype=float)))

    if len(values) == 1:
        return np.array([
            values[0] - 1.0,
            values[0],
            values[0] + 1.0,
        ])

    midpoints = (values[:-1] + values[1:]) / 2.0
    epsilon = max(
        1e-6,
        (values[-1] - values[0]) * 1e-6
    )

    return np.concatenate([
        [values[0] - epsilon],
        midpoints,
        [values[-1] + epsilon],
    ])

def tune_no_answer_threshold(df):
    y_true = df["gold_answerable"].astype(int).to_numpy()
    scores = df["no_answer_score"].to_numpy()

    records = []
    best = None

    for threshold in candidate_thresholds(scores):
        y_pred = predict_answerability(scores, threshold)

        macro_f1 = f1_score(
            y_true,
            y_pred,
            labels=[0, 1],
            average="macro",
            zero_division=0,
        )

        record = {
            "threshold": float(threshold),
            "answerability_macro_f1": float(macro_f1),
        }
        records.append(record)

        if (
            best is None
            or macro_f1 > best["answerability_macro_f1"]
        ):
            best = record

    return best, pd.DataFrame(records)

best_threshold_record, threshold_search = tune_no_answer_threshold(
    raw_predictions
)

NO_ANSWER_THRESHOLD = best_threshold_record["threshold"]

print("Chosen validation threshold:", NO_ANSWER_THRESHOLD)
print(
    "Validation answerability macro-F1:",
    best_threshold_record["answerability_macro_f1"],
)
print(
    f"Decision: no_answer_score > {NO_ANSWER_THRESHOLD:.6f} "
    "means NO ANSWER"
)

Chosen validation threshold: -1.3194911479949951
Validation answerability macro-F1: 0.8046570848703777
Decision: no_answer_score > -1.319491 means NO ANSWER


In [ ]:
def apply_no_answer_threshold(df, threshold):
    output = df.copy()

    output["pred_answerable"] = (
        output["no_answer_score"].to_numpy() <= threshold
    )

    output["prediction"] = np.where(
        output["pred_answerable"],
        output["best_non_empty_answer"],
        "",
    )

    output["exact_match"] = [
        squad_exact_match(pred, gold)
        for pred, gold in zip(
            output["prediction"],
            output["gold_answer"],
        )
    ]

    output["token_f1"] = [
        squad_token_f1(pred, gold)
        for pred, gold in zip(
            output["prediction"],
            output["gold_answer"],
        )
    ]

    return output

predictions = apply_no_answer_threshold(
    raw_predictions,
    NO_ANSWER_THRESHOLD,
)

display(predictions.head())

,id,lang,question,context,gold_answer,gold_answerable,best_non_empty_answer,best_non_empty_start,best_non_empty_end,null_score,best_non_empty_score,no_answer_score,pred_answerable,prediction,exact_match,token_f1
0,0,te,ఒరెగాన్ రాష్ట్రంలోని అతిపెద్ద నగరం ఏది ?,Portland is the largest city in the U.S. state...,Portland,True,Portland,0,8,2.064895,11.935430,-9.870535,True,Portland,1.0,1.0
1,1,te,కలరా వ్యాధిని మొదటగా ఏ దేశంలో కనుగొన్నారు ?,"The word cholera is from ""kholera"" from χολή ""...",Indian subcontinent,True,Indonesia,662,671,0.635047,5.046560,-4.411513,True,Indonesia,0.0,0.0
2,2,te,కలరా వ్యాధిని మొదటగా ఏ దేశంలో కనుగొన్నారు ?,Since it became widespread in the 19th century...,England,True,United States,374,387,1.512879,5.011759,-3.498879,True,United States,0.0,0.0
3,3,te,మొదటి ప్రపంచ యుద్ధం ఎప్పుడు మొదలయింది ?,World War I occurred from 1914 to 1918. In ter...,1914,True,1914 to 1918,26,38,0.565954,11.488725,-10.922771,True,1914 to 1918,0.0,0.5
4,4,te,మొదటి ప్రపంచ యుద్ధం ఎప్పుడు మొదలయింది ?,"World War I (often abbreviated as WWI or WW1),...",28 July 1914,True,28 July 1914 to 11 November 1918,155,187,-0.059871,11.921984,-11.981854,True,28 July 1914 to 11 November 1918,0.0,0.6


In [ ]:
def evaluation_row(df, label):
    y_true = df["gold_answerable"].astype(int).to_numpy()
    y_pred = df["pred_answerable"].astype(int).to_numpy()

    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=[0, 1],   # 0=unanswerable, 1=answerable
        zero_division=0,
    )

    return {
        "language": label,
        "N": len(df),
        "SQuAD_EM": 100 * df["exact_match"].mean(),
        "SQuAD_token_F1": 100 * df["token_f1"].mean(),
        "answerability_macro_F1": 100 * np.mean(f1),

        "unanswerable_precision": 100 * precision[0],
        "unanswerable_recall": 100 * recall[0],
        "unanswerable_F1": 100 * f1[0],

        "answerable_precision": 100 * precision[1],
        "answerable_recall": 100 * recall[1],
        "answerable_F1": 100 * f1[1],
    }

result_rows = [
    evaluation_row(predictions, "overall")
]

for language in ["ar", "ko", "te"]:
    result_rows.append(
        evaluation_row(
            predictions[predictions["lang"] == language],
            language,
        )
    )

results_table = pd.DataFrame(result_rows)
display(results_table.round(2))

,language,N,SQuAD_EM,SQuAD_token_F1,answerability_macro_F1,unanswerable_precision,unanswerable_recall,unanswerable_F1,answerable_precision,answerable_recall,answerable_F1
0,overall,1155,54.89,63.61,80.47,69.33,63.41,66.24,94.03,95.36,94.69
1,ar,415,60.00,68.78,91.54,76.92,96.15,85.47,99.43,95.87,97.62
2,ko,356,55.06,65.63,79.89,51.72,78.95,62.50,98.78,95.85,97.29
3,te,384,49.22,56.17,70.44,69.64,41.94,52.35,83.54,94.16,88.53


In [ ]:
print("Overall answerability classification report:")
print(
    classification_report(
        predictions["gold_answerable"].astype(int),
        predictions["pred_answerable"].astype(int),
        labels=[0, 1],
        target_names=["unanswerable", "answerable"],
        digits=4,
        zero_division=0,
    )
)

print("Confusion matrix")
print("rows = gold [unanswerable, answerable]")
print("cols = predicted [unanswerable, answerable]")
print(
    confusion_matrix(
        predictions["gold_answerable"].astype(int),
        predictions["pred_answerable"].astype(int),
        labels=[0, 1],
    )
)

Overall answerability classification report:
              precision    recall  f1-score   support

unanswerable     0.6933    0.6341    0.6624       164
  answerable     0.9403    0.9536    0.9469       991

    accuracy                         0.9082      1155
   macro avg     0.8168    0.7939    0.8047      1155
weighted avg     0.9052    0.9082    0.9065      1155

Confusion matrix
rows = gold [unanswerable, answerable]
cols = predicted [unanswerable, answerable]
[[104  60]
 [ 46 945]]


In [ ]:
def automatic_error_type(row):
    if row["gold_answerable"] and not row["pred_answerable"]:
        return "false abstention"

    if (
        not row["gold_answerable"]
        and row["pred_answerable"]
    ):
        return "failed abstention"

    if (
        row["gold_answerable"]
        and row["pred_answerable"]
        and row["exact_match"] == 0
    ):
        if row["token_f1"] > 0:
            return "partial-overlap span"
        return "wrong span"

    return "correct"

predictions["automatic_error_type"] = predictions.apply(
    automatic_error_type,
    axis=1,
)

display(
    pd.crosstab(
        predictions["lang"],
        predictions["automatic_error_type"],
        margins=True,
    )
)

error_df = predictions[
    predictions["automatic_error_type"] != "correct"
].copy()

display(
    error_df[
        [
            "id",
            "lang",
            "question",
            "gold_answer",
            "prediction",
            "gold_answerable",
            "pred_answerable",
            "no_answer_score",
            "exact_match",
            "token_f1",
            "automatic_error_type",
        ]
    ].head(20)
)

automatic_error_type,correct,failed abstention,false abstention,partial-overlap span,wrong span,All
lang,,,,,,
ar,249,2,15,77,72,415
ko,196,4,14,82,60,356
te,188,54,17,69,56,384
All,633,60,46,228,188,1155


,id,lang,question,gold_answer,prediction,gold_answerable,pred_answerable,no_answer_score,exact_match,token_f1,automatic_error_type
1,1,te,కలరా వ్యాధిని మొదటగా ఏ దేశంలో కనుగొన్నారు ?,Indian subcontinent,Indonesia,True,True,-4.411513,0.0,0.000000,wrong span
2,2,te,కలరా వ్యాధిని మొదటగా ఏ దేశంలో కనుగొన్నారు ?,England,United States,True,True,-3.498879,0.0,0.000000,wrong span
3,3,te,మొదటి ప్రపంచ యుద్ధం ఎప్పుడు మొదలయింది ?,1914,1914 to 1918,True,True,-10.922771,0.0,0.500000,partial-overlap span
4,4,te,మొదటి ప్రపంచ యుద్ధం ఎప్పుడు మొదలయింది ?,28 July 1914,28 July 1914 to 11 November 1918,True,True,-11.981854,0.0,0.600000,partial-overlap span
7,7,te,మనిషిని సగటున ఉండాల్సిన రక్తం ఎంత?,approximately 5 liters,5 liters,True,True,-12.658011,0.0,0.800000,partial-overlap span
9,9,te,భారతదేశంలో ఓటు హక్కు పొందడానికి ఉండవలసిన కనీస ...,18,21 years to 18 years,True,True,-11.514192,0.0,0.333333,partial-overlap span
12,12,te,భారతదేశంలో సగటుగా ఎన్ని భాషలు మాట్లాడతారు?,"122 which were spoken by more than 10,000 people",122,True,True,-10.923678,0.0,0.200000,partial-overlap span
13,13,te,వాటికన్ సిటి లో పేరుగాంచిన చర్చ్ ఏది?,St. Peter's Basilica,St. Peter,True,True,-5.991092,0.0,0.400000,partial-overlap span
15,15,te,కాలిఫోర్నియాలో రాష్ట్రంలో విస్తీర్ణం పరంగా అతి...,Los Angeles,San Jose,True,True,-8.117379,0.0,0.000000,wrong span
17,17,te,ఉగాండా ఏ ఖండంలో ఉంది?,Africa,,True,False,1.367535,0.0,0.000000,false abstention


In [23]:
manual_parts = []

priority = {
    "false abstention": 0,
    "failed abstention": 1,
    "wrong span": 2,
    "partial-overlap span": 3,
}

for language in ["ar", "ko", "te"]:
    language_errors = error_df[
        error_df["lang"] == language
    ].copy()

    if len(language_errors):
        language_errors["priority"] = (
            language_errors["automatic_error_type"]
            .map(priority)
        )

        language_errors = language_errors.sort_values(
            ["priority", "token_f1"],
            ascending=[True, True],
        ).head(5)

        manual_parts.append(language_errors)

if manual_parts:
    manual_error_analysis = pd.concat(
        manual_parts,
        ignore_index=True,
    )

    manual_error_analysis["manual_category"] = ""
    manual_error_analysis["notes"] = ""

    display(
        manual_error_analysis[
            [
                "id",
                "lang",
                "question",
                "context",
                "gold_answer",
                "prediction",
                "gold_answerable",
                "pred_answerable",
                "automatic_error_type",
                "manual_category",
                "notes",
            ]
        ]
    )

    print(
        "Candidate errors:",
        len(manual_error_analysis),
    )
    print(
        "Per language:",
        manual_error_analysis["lang"].value_counts().to_dict(),
    )
else:
    manual_error_analysis = pd.DataFrame()
    print("No validation errors found.")

,id,lang,question,context,gold_answer,prediction,gold_answerable,pred_answerable,automatic_error_type,manual_category,notes
0,675,ar,هل مصرهي دولة نامية؟,"In recent years, new terms have emerged to des...",Egypt,,True,False,false abstention,,
1,752,ar,هل يعتبر بولس من حواري المسيح؟,"Paul the Apostle (; ; ; c. 5 – c. 64 or 67), c...",apostle,,True,False,false abstention,,
2,756,ar,من أول قوة عسكرية إستعملت جهاز الاتصال أول مرة؟,The alert measurement systems are various stat...,U.S. Armed Forces,,True,False,false abstention,,
3,773,ar,هل بنى النبيّ سليمان القدس؟,The Hebrew Bible credits him as the builder of...,The Hebrew Bible credits him as the builder of...,,True,False,false abstention,,
4,787,ar,هل صلاة الفجر سنة أم فرض على العبد ؟,"The Fajr prayer ( """", ""dawn prayer"") an obliga...",obligatory,,True,False,false abstention,,
5,286,ko,옴진리교는 어느 나라에서 시작된 종교인가?,These letters are believed to have derived fro...,Egypt,,True,False,false abstention,,
6,385,ko,고대 사람들은 어떤 방법으로 행성을 관측했는가?,Ancient astronomers were able to differentiate...,"differentiate between stars and planets, as st...",,True,False,false abstention,,
7,399,ko,소아마비 바이러스의 증상은?,"Poliomyelitis, often called polio or infantile...",muscle weakness resulting in an inability to move,,True,False,false abstention,,
8,431,ko,지자기 폭풍이 지구에 일어났을때 지구에선 어떤 현상이 일어나는가?,Geomagnetic storms and increased solar ultravi...,"heat Earth's upper atmosphere, causing it to e...",,True,False,false abstention,,
9,436,ko,고대 사람들은 어떤 방법으로 행성을 관측했는가?,Early cultures identified celestial oFAncient ...,Ancient structures with possibly astronomical ...,,True,False,false abstention,,


Candidate errors: 15
Per language: {'ar': 5, 'ko': 5, 'te': 5}


In [15]:
# Run after manually filling manual_category above.
if len(manual_error_analysis):
    filled = manual_error_analysis[
        manual_error_analysis["manual_category"]
        .astype(str)
        .str.strip() != ""
    ]

    print("Manually categorized:", len(filled))

    if len(filled):
        display(
            pd.crosstab(
                filled["manual_category"],
                filled["lang"],
                margins=True,
            )
        )

    if len(filled) < 10:
        print(
            "WARNING: Week 40–41 requires at least "
            "10 manually analysed validation errors."
        )

    if not {"ar", "ko", "te"}.issubset(
        set(filled["lang"])
    ):
        print(
            "WARNING: The analysis must include "
            "Arabic, Korean, and Telugu."
        )

Manually categorized: 0
